# Chapter 10. Pitch and formant quality control

**Learning objective.** Treat F0 and formant outputs as profile-dependent estimates, preserve every requested observation and failure, and choose settings from blinded signal-level diagnostics rather than group outcomes.

**Estimated time:** 90 minutes. **Exercises:** 10.1 and 10.2. The six bundled WAV files are explicitly **synthetic teaching fixtures** with known generator parameters. The included autocorrelation and spectral-peak estimators are teaching implementations, not validated production measurement systems.

## Input and output contract

Inputs are source and interval manifests, a completed measurement specification, and generated PCM audio. Outputs are long-form raw candidates, flags, an HTML settings comparison, a blinded review sample, a measure-specific gate, and run provenance under `companion/outputs/ch10/01_pitch_and_formant_qc/`. Source audio is hash-checked and never overwritten.

In [ ]:
from pathlib import Path
import json
import sys

start = Path.cwd().resolve()
for candidate in (start, *start.parents):
    if (candidate / "companion" / "data" / "MANIFEST.tsv").is_file():
        REPOSITORY_ROOT = candidate
        break
else:
    raise RuntimeError("Run from the repository root or one of its subdirectories.")
SOURCE_ROOT = REPOSITORY_ROOT / "companion" / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
DATA_ROOT = REPOSITORY_ROOT / "companion" / "data" / "ch10" / "synthetic_acoustics"
if not DATA_ROOT.exists():
    raise FileNotFoundError("Chapter 10 fixture absent. Run companion/scripts/generate_ch10_fixture.py explicitly; no data are substituted silently.")
print("DATA STATUS: SYNTHETIC_TEACHING_FIXTURE — generated signals and landmarks, not human speech.")

In [ ]:
from phonetic_research_companion.ch10 import run_pitch_and_formant_qc
OUTPUT = REPOSITORY_ROOT / "companion" / "outputs" / "ch10" / "01_pitch_and_formant_qc"
result = run_pitch_and_formant_qc(DATA_ROOT / "source_manifest.tsv", DATA_ROOT / "interval_manifest.tsv", DATA_ROOT / "acoustic_measurement_spec.yaml", OUTPUT)
assert result["raw"]
assert result["flags"]
assert {row["measure"] for row in result["raw"]} == {"f0", "formant"}
print("Raw estimates:", len(result["raw"]), "Flags:", len(result["flags"]), "Review rows:", len(result["review"]))
for summary in result["summaries"]:
    print(summary)

## Learner gate

Compare the prespecified floor–ceiling and maximum-formant profiles without inspecting the generator role or any group outcome. Explain missing frames, search-limit contact, abrupt jumps, and insufficient peaks. For each measure, choose **accept**, **restrict**, **amend and rerun**, or **reject**, and state why that decision would not automatically generalize to natural speech.

In [ ]:
learner_gate = {"data_status": "SYNTHETIC_TEACHING_FIXTURE", "decision_status": "undecided", "f0_gate": "Learner response required.", "formant_gate": "Learner response required.", "scope": "synthetic teaching signals only", "warning": "Generated-signal performance is not validation on human speech."}
(OUTPUT / "learner_measurement_gate.json").write_text(json.dumps(learner_gate, indent=2), encoding="utf-8")
required = ["raw_acoustic_estimates.tsv", "measurement_flags.tsv", "settings_comparison.html", "review_sample.tsv", "measurement_gate.json", "run_provenance.json", "learner_measurement_gate.json"]
assert all((OUTPUT / name).exists() for name in required)
print(json.dumps(learner_gate, indent=2))